# Base de Donnees Mondiale des Centrales Electriques
## Analyse Complete : NumPy · Pandas · Matplotlib · Seaborn




## 0. Imports et Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import f_oneway, kruskal, ttest_ind
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Style global dark
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1f2e',
    'axes.edgecolor':   '#2e3448',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#e0e0e0',
    'ytick.color':      '#e0e0e0',
    'text.color':       '#e0e0e0',
    'grid.color':       '#2e3448',
    'grid.alpha':       0.55,
    'legend.facecolor': '#1a1f2e',
    'legend.edgecolor': '#2e3448',
})

PALETTE = {
    'Solar':      '#FFD54F',
    'Hydro':      '#4FC3F7',
    'Wind':       '#81C784',
    'Gas':        '#FF8A65',
    'Coal':       '#B0BEC5',
    'Oil':        '#CE93D8',
    'Biomass':    '#80DEEA',
    'Waste':      '#FFAB40',
    'Nuclear':    '#EF9A9A',
    'Geothermal': '#A5D6A7',
    'Other':      '#F48FB1',
}

print(f"numpy      {np.__version__}")
print(f"pandas     {pd.__version__}")
print(f"matplotlib {plt.matplotlib.__version__}")
print(f"seaborn    {sns.__version__}")


## Tache 1 — Importation et Nettoyage des Donnees

**Etapes :**
- Chargement du CSV avec Pandas
- Identification et traitement des valeurs manquantes
- Conversion NumPy des colonnes numeriques
- Creation de features derivees utiles


In [ ]:
# Chargement
df_raw = pd.read_csv('power_plants.csv', low_memory=False)
print(f"Dimensions brutes : {df_raw.shape[0]:,} centrales x {df_raw.shape[1]} colonnes")
print(f"\nColonnes : {df_raw.columns.tolist()}")
df_raw.head(3)


In [ ]:
# --- Valeurs manquantes ---
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
missing_df = pd.DataFrame({'Manquant': missing, 'Pct (%)': missing_pct})
missing_df = missing_df[missing_df['Manquant'] > 0].sort_values('Pct (%)', ascending=False)
print("Colonnes avec valeurs manquantes :")
print(missing_df.to_string())


In [ ]:
# --- Nettoyage ---
df = df_raw.copy()

# Colonnes de generation annuelle
GEN_COLS = ['generation_gwh_2013','generation_gwh_2014','generation_gwh_2015',
            'generation_gwh_2016','generation_gwh_2017','generation_gwh_2018',
            'generation_gwh_2019']

# 1. Conversion NumPy : forcer les types numeriques
for col in ['capacity_mw','latitude','longitude','commissioning_year'] + GEN_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # Remplacement par np.nan (standard NumPy)
    df[col] = np.where(df[col] < 0, np.nan, df[col])

# 2. Imputation strategique
#    capacity_mw : mediane par type de carburant
df['capacity_mw'] = df.groupby('primary_fuel')['capacity_mw'].transform(
    lambda x: x.fillna(x.median())
)

# 3. Regroupement des carburants rares -> 'Other'
main_fuels = ['Solar','Hydro','Wind','Gas','Coal','Oil','Biomass','Waste','Nuclear','Geothermal']
df['fuel_group'] = np.where(df['primary_fuel'].isin(main_fuels),
                            df['primary_fuel'], 'Other')

# 4. Feature : generation moyenne disponible (moy. sur annees renseignees)
df['gen_mean_gwh'] = df[GEN_COLS].mean(axis=1)

# 5. Feature : decennie de mise en service
df['decade'] = (df['commissioning_year'] // 10 * 10).astype('Int64')

# 6. Feature : log capacite (pour visualisations)
df['log_capacity'] = np.log1p(df['capacity_mw'])

print(f"Dimensions apres nettoyage : {df.shape}")
print(f"Valeurs manquantes restantes (colonnes cles) :")
print(df[['capacity_mw','primary_fuel','latitude','longitude']].isnull().sum())
print(f"\nTypes de carburant apres regroupement :")
print(df['fuel_group'].value_counts())


## Tache 2 — Analyse Exploratoire des Donnees (EDA)

In [ ]:
# --- Statistiques globales ---
print("=" * 55)
print("STATISTIQUES GLOBALES")
print("=" * 55)
print(f"  Nombre de centrales     : {len(df):,}")
print(f"  Pays couverts           : {df['country_long'].nunique()}")
print(f"  Types de carburant      : {df['primary_fuel'].nunique()}")
print(f"  Capacite totale (GW)    : {df['capacity_mw'].sum()/1000:,.1f}")
print(f"  Capacite moyenne (MW)   : {df['capacity_mw'].mean():.1f}")
print(f"  Capacite mediane (MW)   : {df['capacity_mw'].median():.1f}")

# --- Statistiques NumPy sur capacity_mw ---
cap = df['capacity_mw'].dropna().values
print(f"\n--- NumPy sur capacity_mw ---")
print(f"  np.mean   : {np.mean(cap):.2f} MW")
print(f"  np.median : {np.median(cap):.2f} MW")
print(f"  np.std    : {np.std(cap):.2f} MW")
print(f"  np.percentile [25,50,75,90,99] : {np.percentile(cap,[25,50,75,90,99]).round(1)}")
print(f"  np.max    : {np.max(cap):,.0f} MW")
print(f"  Asymetrie : {stats.skew(cap):.4f}  (forte droite = grosses centrales rares)")


In [ ]:
# --- Top 15 pays par capacite totale ---
top_countries = (df.groupby('country_long')['capacity_mw']
                   .sum()
                   .sort_values(ascending=False)
                   .head(15) / 1000)   # en GW

print("Top 15 pays par capacite installee (GW) :")
print(top_countries.round(1).to_string())

# --- Repartition par type de carburant ---
fuel_stats = df.groupby('fuel_group')['capacity_mw'].agg(
    Nb_centrales = 'count',
    Capacite_GW  = lambda x: x.sum() / 1000,
    Capacite_moy_MW = 'mean',
    Capacite_med_MW = 'median',
).round(2).sort_values('Capacite_GW', ascending=False)

print("\nRepartition par type de carburant :")
display(fuel_stats)


## Tache 3 — Analyse Statistique par Type de Carburant



In [ ]:
# --- Groupes de capacite par carburant (NumPy) ---
top_fuels = ['Coal','Gas','Nuclear','Hydro','Wind','Solar']
groups = {}
for fuel in top_fuels:
    arr = df[df['fuel_group'] == fuel]['capacity_mw'].dropna().values
    groups[fuel] = arr
    print(f"  {fuel:<12} n={len(arr):5,}  moy={np.mean(arr):8.1f} MW  "
          f"med={np.median(arr):7.1f} MW  std={np.std(arr):8.1f}")

# --- ANOVA a un facteur ---
F_stat, p_anova = f_oneway(*[groups[f] for f in top_fuels])
H_stat, p_kruskal = kruskal(*[groups[f] for f in top_fuels])

print(f"\nANOVA    F={F_stat:.2f}  p={p_anova:.2e}")
print(f"Kruskal  H={H_stat:.2f}  p={p_kruskal:.2e}")
if p_anova < 0.05:
    print("-> Les capacites moyennes different SIGNIFICATIVEMENT selon le type de carburant (p<0.05)")

# --- Test T : Coal vs Nuclear ---
t, p = ttest_ind(groups['Coal'], groups['Nuclear'])
print(f"\nTest T  Coal vs Nuclear : t={t:.2f}  p={p:.2e}")
print(f"  Coal moy={np.mean(groups['Coal']):.1f} MW  vs  Nuclear moy={np.mean(groups['Nuclear']):.1f} MW")


In [ ]:
# --- Intervalles de confiance (NumPy) ---
print("Intervalles de confiance a 95% de la capacite moyenne par carburant :")
print(f"  {'Carburant':<12}  {'Moy (MW)':>10}  {'IC bas':>10}  {'IC haut':>10}")
print("  " + "-"*48)
for fuel in top_fuels:
    arr  = groups[fuel]
    n    = len(arr)
    mean = np.mean(arr)
    se   = np.std(arr, ddof=1) / np.sqrt(n)
    margin = stats.t.ppf(0.975, df=n-1) * se
    print(f"  {fuel:<12}  {mean:>10.1f}  {mean-margin:>10.1f}  {mean+margin:>10.1f}")


## Tache 4 — Analyse des Series Chronologiques




In [ ]:
# Dataset avec annee de mise en service connue
df_ts = df[df['commissioning_year'].notna() &
           (df['commissioning_year'] >= 1960) &
           (df['commissioning_year'] <= 2020)].copy()
df_ts['year'] = df_ts['commissioning_year'].astype(int)

print(f"Centrales avec annee connue (1960-2020) : {len(df_ts):,}")

# --- Tendance annuelle : nb de centrales + capacite installee ---
yearly = df_ts.groupby('year').agg(
    nb_centrales  = ('capacity_mw', 'count'),
    capacite_GW   = ('capacity_mw', lambda x: x.sum() / 1000),
).reset_index()

# Moyenne mobile 5 ans (NumPy)
yearly['nb_rolling5']  = np.convolve(yearly['nb_centrales'],
                                     np.ones(5)/5, mode='same')
yearly['cap_rolling5'] = np.convolve(yearly['capacite_GW'],
                                     np.ones(5)/5, mode='same')

# --- Mix de carburant par decennie ---
decade_fuel = (df_ts.groupby(['decade','fuel_group'])
                    .agg(nb=('capacity_mw','count'),
                         cap=('capacity_mw', lambda x: x.sum()/1000))
                    .reset_index())

top_d_fuels = ['Coal','Gas','Nuclear','Hydro','Wind','Solar','Oil','Biomass']
pivot_cap = (decade_fuel[decade_fuel['fuel_group'].isin(top_d_fuels)]
                .pivot_table(index='decade', columns='fuel_group',
                             values='cap', fill_value=0))

print("\nCapacite installee (GW) par decennie et type de carburant :")
display(pivot_cap.round(1))

# --- Taux de croissance annuel moyen (NumPy) ---
cap_arr = yearly['capacite_GW'].values
pct_change = np.diff(cap_arr) / cap_arr[:-1] * 100
print(f"\nCroissance capacite annuelle moyenne 1960-2020 : {np.nanmean(pct_change):.1f}%")
print(f"Annee record (plus forte capacite ajoutee)    : {yearly.loc[yearly['capacite_GW'].idxmax(),'year']}")


## Tache 5 — Visualisations Avancees

### 5.1 Tableau de bord principal

In [ ]:
fig = plt.figure(figsize=(22, 17))
fig.patch.set_facecolor('#0f1117')
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.38)

pal = [PALETTE.get(f,'#90A4AE') for f in df['fuel_group'].value_counts().index]

# ── 1. Repartition des centrales par type de carburant (donut) ────────────
ax1 = fig.add_subplot(gs[0, 0])
fuel_cnt = df['fuel_group'].value_counts()
wedges, texts, autotexts = ax1.pie(
    fuel_cnt.values, labels=fuel_cnt.index,
    colors=[PALETTE.get(f,'#90A4AE') for f in fuel_cnt.index],
    autopct='%1.1f%%', startangle=90, pctdistance=0.80,
    wedgeprops=dict(width=0.55),
    textprops={'color':'#e0e0e0','fontsize':8})
for at in autotexts: at.set_fontsize(7)
ax1.set_title('Repartition par type de carburant', fontweight='bold', fontsize=11)

# ── 2. Top 12 pays par capacite ───────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
top12 = df.groupby('country_long')['capacity_mw'].sum().nlargest(12) / 1000
colors12 = plt.cm.plasma(np.linspace(0.2, 0.9, 12))
bars = ax2.barh(top12.index[::-1], top12.values[::-1],
                color=colors12, edgecolor='#0f1117', alpha=0.88)
for b in bars:
    ax2.text(b.get_width()+5, b.get_y()+b.get_height()/2,
             f'{b.get_width():.0f} GW', va='center', fontsize=8, color='#FFD54F')
ax2.set_title('Top 12 pays — Capacite totale (GW)', fontweight='bold', fontsize=11)
ax2.set_xlabel('Capacite (GW)')

# ── 3. Distribution log(capacite) par carburant (violin) ─────────────────
ax3 = fig.add_subplot(gs[0, 2])
v_fuels = ['Coal','Gas','Nuclear','Hydro','Wind','Solar']
v_data  = [df[df['fuel_group']==f]['log_capacity'].dropna().values for f in v_fuels]
parts   = ax3.violinplot(v_data, positions=range(1,7), showmedians=True)
for pc, fuel in zip(parts['bodies'], v_fuels):
    pc.set_facecolor(PALETTE.get(fuel,'#90A4AE')); pc.set_alpha(0.68)
parts['cmedians'].set_color('white'); parts['cmedians'].set_lw(2)
ax3.set_xticks(range(1,7)); ax3.set_xticklabels(v_fuels, fontsize=8, rotation=20)
ax3.set_title('Distribution log(Capacite) par carburant', fontweight='bold', fontsize=11)
ax3.set_ylabel('log(1 + capacite MW)')

# ── 4. Serie temporelle : nb centrales par an ─────────────────────────────
ax4 = fig.add_subplot(gs[1, :2])
ax4.fill_between(yearly['year'], yearly['nb_centrales'], alpha=0.25, color='#4FC3F7')
ax4.plot(yearly['year'], yearly['nb_centrales'], color='#4FC3F7', lw=1.2, alpha=0.6)
ax4.plot(yearly['year'], yearly['nb_rolling5'], color='#FF8A65', lw=2.5,
         label='Moy. mobile 5 ans')
ax4.set_title('Nouvelles centrales mises en service par annee (1960-2020)',
              fontweight='bold', fontsize=11)
ax4.set_xlabel('Annee'); ax4.set_ylabel('Nb centrales')
ax4.legend(fontsize=9)

# ── 5. Mix energetique par decennie (stacked bar) ─────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
dec_order = sorted(pivot_cap.index.dropna())
pivot_plot = pivot_cap.loc[[d for d in dec_order if d in pivot_cap.index]]
bottom_ = np.zeros(len(pivot_plot))
for fuel in [c for c in top_d_fuels if c in pivot_plot.columns]:
    vals = pivot_plot[fuel].values
    ax5.bar(range(len(pivot_plot)), vals, bottom=bottom_,
            color=PALETTE.get(fuel,'#90A4AE'), label=fuel,
            edgecolor='#0f1117', alpha=0.88, lw=0.4, width=0.8)
    bottom_ += vals
ax5.set_xticks(range(len(pivot_plot)))
ax5.set_xticklabels([str(int(d)) for d in pivot_plot.index], rotation=40, fontsize=8)
ax5.set_title('Mix energetique par decennie (GW)', fontweight='bold', fontsize=11)
ax5.set_ylabel('Capacite (GW)')
ax5.legend(fontsize=7, ncol=2, loc='upper left')

# ── 6. Box plot capacite par carburant ────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 0])
box_data = [np.log1p(df[df['fuel_group']==f]['capacity_mw'].dropna().values)
            for f in v_fuels]
bp = ax6.boxplot(box_data, patch_artist=True, vert=True,
                 medianprops=dict(color='white', lw=2),
                 whiskerprops=dict(color='#e0e0e0'),
                 capprops=dict(color='#e0e0e0'),
                 flierprops=dict(marker='.', alpha=0.2, color='#e0e0e0', ms=3))
for patch, fuel in zip(bp['boxes'], v_fuels):
    patch.set_facecolor(PALETTE.get(fuel,'#90A4AE')); patch.set_alpha(0.72)
ax6.set_xticks(range(1,7)); ax6.set_xticklabels(v_fuels, fontsize=8, rotation=20)
ax6.set_title('Boxplot log(Capacite) par carburant', fontweight='bold', fontsize=11)
ax6.set_ylabel('log(1 + capacite MW)')

# ── 7. Heatmap correlation numerique ──────────────────────────────────────
ax7 = fig.add_subplot(gs[2, 1])
num_cols = ['capacity_mw','latitude','longitude','gen_mean_gwh','log_capacity']
corr = df[num_cols].dropna().corr()
sns.heatmap(corr, ax=ax7, cmap='coolwarm', vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size':8},
            linewidths=0.4, linecolor='#0f1117',
            cbar_kws={'shrink':0.8})
ax7.set_title('Matrice de correlation (NumPy)', fontweight='bold', fontsize=11)
ax7.tick_params(axis='x', rotation=30, labelsize=8)
ax7.tick_params(axis='y', rotation=0,  labelsize=8)

# ── 8. Scatter capacite vs generation ─────────────────────────────────────
ax8 = fig.add_subplot(gs[2, 2])
sub = df[df['gen_mean_gwh'].notna() & df['fuel_group'].isin(v_fuels)].sample(2000, random_state=1)
for fuel in v_fuels:
    s = sub[sub['fuel_group']==fuel]
    ax8.scatter(np.log1p(s['capacity_mw']), np.log1p(s['gen_mean_gwh']),
                c=PALETTE.get(fuel,'#90A4AE'), s=12, alpha=0.55,
                label=fuel, edgecolors='none')
ax8.set_title('log(Capacite) vs log(Generation moy.)', fontweight='bold', fontsize=11)
ax8.set_xlabel('log(1+Capacite MW)')
ax8.set_ylabel('log(1+Generation GWh)')
ax8.legend(fontsize=7, markerscale=2)

fig.suptitle('Tableau de Bord — Base de Donnees Mondiale des Centrales Electriques',
             color='#e0e0e0', fontsize=15, fontweight='bold', y=1.005)
plt.savefig('power_dashboard.png', dpi=140, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("Tableau de bord principal genere")


### 5.2 Carte geographique des centrales

In [ ]:
fig2, axes = plt.subplots(1, 2, figsize=(18, 7))
fig2.patch.set_facecolor('#0f1117')

# --- Carte mondiale (projection simple lat/lon) ---
ax = axes[0]
ax.set_facecolor('#0d1b2a')

# Fond : rectangle ocean
from matplotlib.patches import Rectangle
ax.add_patch(Rectangle((-180,-90), 360, 180, color='#0d1b2a', zorder=0))

# Points des centrales colores par carburant
sample_geo = df[df['latitude'].notna() & df['longitude'].notna()].copy()

for fuel in list(PALETTE.keys())[::-1]:
    s = sample_geo[sample_geo['fuel_group'] == fuel]
    size = np.clip(np.sqrt(s['capacity_mw']) * 0.15, 0.5, 8)
    ax.scatter(s['longitude'], s['latitude'],
               c=PALETTE[fuel], s=size, alpha=0.45,
               edgecolors='none', label=fuel, rasterized=True)

ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
ax.set_title('Distribution Geographique des Centrales Electriques',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.legend(fontsize=7, ncol=2, loc='lower left',
          markerscale=2, framealpha=0.7)
ax.grid(True, alpha=0.15, color='white', linestyle=':')

# --- Zoom : densite par carburant (heat approche) ---
ax2 = axes[1]
for fuel in ['Coal','Nuclear','Solar','Wind','Hydro']:
    s = sample_geo[sample_geo['fuel_group'] == fuel]
    ax2.scatter(s['longitude'], s['latitude'],
                c=PALETTE[fuel], s=3, alpha=0.35,
                edgecolors='none', label=fuel)
ax2.set_xlim(-180, 180); ax2.set_ylim(-60, 80)
ax2.set_facecolor('#0d1b2a')
ax2.set_title('Zoom : Carburants majeurs (Coal, Nuclear, Solar, Wind, Hydro)',
              fontweight='bold', fontsize=12)
ax2.set_xlabel('Longitude'); ax2.set_ylabel('Latitude')
ax2.legend(fontsize=8, markerscale=3)
ax2.grid(True, alpha=0.15, color='white', linestyle=':')

plt.suptitle('Repartition Geographique Mondiale des Centrales',
             color='#e0e0e0', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('power_map.png', dpi=140, bbox_inches='tight', facecolor='#0f1117')
plt.show()


## Tache 6 — Operations Matricielles et Valeurs Propres (PCA)




In [ ]:
# --- Preparation de la matrice de features ---
feat_cols = ['capacity_mw','latitude','longitude','log_capacity']
gen_available = [c for c in ['generation_gwh_2017','generation_gwh_2018','generation_gwh_2019']
                 if c in df.columns]
feat_cols += gen_available

df_feat = df[feat_cols + ['fuel_group']].dropna(subset=feat_cols)
X = df_feat[feat_cols].values

# Standardisation (NumPy + sklearn)
X_std = StandardScaler().fit_transform(X)
print(f"Matrice X standardisee : {X_std.shape}")

# --- Matrice de covariance (NumPy) ---
cov_matrix = np.cov(X_std.T)
print(f"\nMatrice de covariance ({cov_matrix.shape[0]}x{cov_matrix.shape[1]}) :")
print(np.round(cov_matrix, 3))

# --- Valeurs propres et vecteurs propres (NumPy) ---
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

# Tri decroissant
idx_sort      = np.argsort(eigenvalues)[::-1]
eigenvalues   = eigenvalues[idx_sort].real
eigenvectors  = eigenvectors[:, idx_sort].real

explained_var = eigenvalues / eigenvalues.sum() * 100
cumvar        = np.cumsum(explained_var)

print("\nValeurs propres (variance expliquee) :")
for i, (ev, pct, cum) in enumerate(zip(eigenvalues, explained_var, cumvar)):
    print(f"  PC{i+1}: λ={ev:.4f}  ({pct:.2f}%  cumulé: {cum:.2f}%)")

print(f"\nNombre de composantes pour 80% de variance : {np.argmax(cumvar >= 80) + 1}")
print(f"Nombre de composantes pour 95% de variance : {np.argmax(cumvar >= 95) + 1}")


In [ ]:
# --- ACP avec sklearn pour projection ---
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_std)

print("Contribution des features aux 3 premieres composantes :")
loadings = pd.DataFrame(
    pca.components_.T,
    index=feat_cols,
    columns=[f'PC{i+1}' for i in range(3)]
).round(4)
display(loadings)

# --- Visualisations PCA ---
fig3, axes = plt.subplots(1, 3, figsize=(18, 6))
fig3.patch.set_facecolor('#0f1117')

# Variance expliquee
axes[0].bar(range(1, len(eigenvalues)+1), explained_var,
            color='#4FC3F7', alpha=0.85, edgecolor='#0f1117')
ax_t = axes[0].twinx()
ax_t.plot(range(1, len(eigenvalues)+1), cumvar,
          'o-', color='#FF8A65', lw=2, ms=6)
ax_t.axhline(80, color='#FFD54F', lw=1.5, linestyle='--', alpha=0.7)
ax_t.set_ylabel('Cumulé (%)', color='#FF8A65', fontsize=9)
ax_t.tick_params(colors='#FF8A65'); ax_t.set_facecolor('#1a1f2e')
axes[0].set_title('Valeurs propres — Variance expliquee', fontweight='bold')
axes[0].set_xlabel('Composante principale')
axes[0].set_ylabel('Variance (%)')

# Scatter PC1 vs PC2
fuels_pca = df_feat['fuel_group'].values
for fuel in list(PALETTE.keys())[:8]:
    mask = fuels_pca == fuel
    if mask.sum() > 0:
        axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                        c=PALETTE[fuel], s=8, alpha=0.45,
                        label=fuel, edgecolors='none')
axes[1].set_title(f'ACP PC1 vs PC2
({pca.explained_variance_ratio_[:2].sum()*100:.1f}% variance)',
                  fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].legend(fontsize=7, markerscale=2)

# Heatmap des loadings
sns.heatmap(loadings.T, ax=axes[2], cmap='RdBu_r', center=0,
            annot=True, fmt='.2f', annot_kws={'size':8},
            linewidths=0.3, cbar_kws={'shrink':0.8})
axes[2].set_title('Loadings — contribution des features', fontweight='bold')
axes[2].tick_params(axis='x', rotation=30, labelsize=8)

fig3.suptitle('Tache 6 — Operations Matricielles et ACP',
              color='#e0e0e0', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## Tache 7 — Integration NumPy + Pandas + Matplotlib

Demonstration de l'usage conjoint des trois bibliotheques pour des
analyses et visualisations impossibles avec une seule librairie.


In [ ]:
# ── 7.1 Filtrage complexe NumPy dans Pandas ──────────────────────────────
print("7.1 Filtrage NumPy dans Pandas")
print("-" * 45)

# Identifier les centrales au-dessus du 95e percentile de capacite PAR carburant
cap_arr   = df['capacity_mw'].values
fuel_arr  = df['fuel_group'].values
mask_giant = np.zeros(len(df), dtype=bool)

for fuel in main_fuels:
    idx_fuel  = np.where(fuel_arr == fuel)[0]
    if len(idx_fuel) == 0: continue
    cap_fuel  = cap_arr[idx_fuel]
    p95       = np.nanpercentile(cap_fuel, 95)
    mask_giant[idx_fuel[cap_fuel >= p95]] = True

giants = df[mask_giant].copy()
print(f"Centrales 'geantes' (>P95 par carburant) : {mask_giant.sum():,}")
print(giants.groupby('fuel_group')[['capacity_mw','country_long']].agg(
    {'capacity_mw': ['count','mean','max'], 'country_long': 'nunique'}
).round(1))

# ── 7.2 Calcul de distances geographiques (NumPy vectorise) ──────────────
print("\n7.2 Distance geographique vectorisee (NumPy haversine)")
print("-" * 50)

def haversine_np(lat1, lon1, lat2, lon2):
    R    = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a    = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# Distance de chaque centrale a Paris (48.85, 2.35)
lat_paris, lon_paris = 48.85, 2.35
df_geo = df[df['latitude'].notna() & df['longitude'].notna()].copy()
df_geo['dist_paris_km'] = haversine_np(
    df_geo['latitude'].values, df_geo['longitude'].values,
    lat_paris, lon_paris
)
print(f"Centrale la plus proche de Paris : {df_geo.loc[df_geo['dist_paris_km'].idxmin(),'name']}")
print(f"  Distance : {df_geo['dist_paris_km'].min():.1f} km")
print(f"Centrale la plus eloignee de Paris : {df_geo.loc[df_geo['dist_paris_km'].idxmax(),'name']}")
print(f"  Distance : {df_geo['dist_paris_km'].max():.0f} km")


In [ ]:
# ── 7.3 Graphique sophistique NumPy + Matplotlib ─────────────────────────
print("7.3 Visualisations avancees NumPy + Matplotlib")

fig4, axes = plt.subplots(2, 2, figsize=(16, 12))
fig4.patch.set_facecolor('#0f1117')

# --- a) Evolution part renouvelable / fossile (NumPy sur pivot) ──────────
ax = axes[0, 0]
renew  = ['Solar','Wind','Hydro','Biomass','Geothermal','Waste']
fossil = ['Coal','Gas','Oil','Petcoke','Cogeneration']

piv_yr = df_ts.groupby(['year','fuel_group'])['capacity_mw'].sum().unstack(fill_value=0)
years_arr = piv_yr.index.values

r_cols = [c for c in renew if c in piv_yr.columns]
f_cols = [c for c in fossil if c in piv_yr.columns]

r_total = piv_yr[r_cols].values.sum(axis=1)
f_total = piv_yr[f_cols].values.sum(axis=1)
total   = r_total + f_total + 1e-9
r_pct   = r_total / total * 100
f_pct   = f_total / total * 100

# Moyenne mobile NumPy (convolution)
r_smooth = np.convolve(r_pct, np.ones(5)/5, mode='same')
f_smooth = np.convolve(f_pct, np.ones(5)/5, mode='same')

ax.fill_between(years_arr, r_smooth, alpha=0.35, color='#81C784', label='Renouvelable')
ax.fill_between(years_arr, f_smooth, alpha=0.35, color='#FF8A65', label='Fossile')
ax.plot(years_arr, r_smooth, color='#81C784', lw=2)
ax.plot(years_arr, f_smooth, color='#FF8A65', lw=2)
ax.set_title('Part renouvelable vs fossile (% nb centrales, moy. 5 ans)',
             fontweight='bold')
ax.set_xlabel('Annee'); ax.set_ylabel('Part (%)')
ax.legend(fontsize=9)

# --- b) Histogramme capacite log-normal + fit theorique ──────────────────
ax = axes[0, 1]
cap_log = np.log10(df['capacity_mw'].dropna().values + 1)
n_hist, bins, patches = ax.hist(cap_log, bins=60, density=True,
                                color='#4FC3F7', edgecolor='#0f1117', alpha=0.75)
# Fit gaussien (NumPy)
mu_fit, std_fit = np.mean(cap_log), np.std(cap_log)
x_fit = np.linspace(bins[0], bins[-1], 300)
gauss = (1/(std_fit*np.sqrt(2*np.pi))) * np.exp(-((x_fit-mu_fit)**2)/(2*std_fit**2))
ax.plot(x_fit, gauss, color='#FF8A65', lw=2.5, label=f'N(μ={mu_fit:.2f}, σ={std_fit:.2f})')
ax.set_title('Distribution log10(capacite) + fit gaussien', fontweight='bold')
ax.set_xlabel('log10(Capacite MW + 1)')
ax.set_ylabel('Densite')
ax.legend(fontsize=9)

# --- c) Distance a Paris vs capacite ────────────────────────────────────
ax = axes[1, 0]
for fuel in ['Coal','Nuclear','Solar','Wind']:
    s = df_geo[df_geo['fuel_group']==fuel].sample(min(300,len(df_geo[df_geo['fuel_group']==fuel])),
                                                   random_state=0)
    ax.scatter(s['dist_paris_km'], np.log1p(s['capacity_mw']),
               c=PALETTE[fuel], s=10, alpha=0.5, label=fuel, edgecolors='none')
ax.set_title('Distance a Paris vs log(Capacite)', fontweight='bold')
ax.set_xlabel('Distance a Paris (km)')
ax.set_ylabel('log(1+Capacite MW)')
ax.legend(fontsize=8, markerscale=2)

# --- d) Tendance capacite moyenne + IC (NumPy bootstrapping) ─────────────
ax = axes[1, 1]
np.random.seed(42)
year_range = range(1970, 2021)
boot_means, boot_lo, boot_hi = [], [], []
for yr in year_range:
    arr = df_ts[df_ts['year']==yr]['capacity_mw'].dropna().values
    if len(arr) < 5:
        boot_means.append(np.nan); boot_lo.append(np.nan); boot_hi.append(np.nan)
        continue
    samples = np.array([np.mean(np.random.choice(arr, len(arr), replace=True))
                        for _ in range(200)])
    boot_means.append(np.mean(arr))
    boot_lo.append(np.percentile(samples, 2.5))
    boot_hi.append(np.percentile(samples, 97.5))

yr_arr = np.array(list(year_range))
bm, bl, bh = np.array(boot_means), np.array(boot_lo), np.array(boot_hi)
valid = ~np.isnan(bm)
ax.plot(yr_arr[valid], bm[valid], color='#FFD54F', lw=2.5, label='Moy. capacite')
ax.fill_between(yr_arr[valid], bl[valid], bh[valid], alpha=0.3, color='#FFD54F',
                label='IC 95% (bootstrap)')
ax.set_title('Capacite moyenne annuelle + IC Bootstrap (NumPy)', fontweight='bold')
ax.set_xlabel('Annee'); ax.set_ylabel('Capacite moyenne (MW)')
ax.legend(fontsize=9)

fig4.suptitle('Tache 7 — Integration NumPy + Pandas + Matplotlib',
              color='#e0e0e0', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('power_integration.png', dpi=140, bbox_inches='tight', facecolor='#0f1117')
plt.show()
